# APIs and Multiprocessing

*This notebook includes adapted content from [Melanie Walsh's chapter on Data Collection](https://melaniewalsh.github.io/Intro-Cultural-Analytics/04-Data-Collection/00-Data-Collection.html).*

In this lab, we'll introduce a useful way to extract data. We'll go over the following topics:

- Accessing an API
- API Wrappers
- Python Scripts
- Multiprocessing

# APIs

**An API is an Application Programming Interface that allows you to programmatically extract and interact with company data which drives their websites.** In this way, social networks, museums, foundations, research labs, applications, and projects can make their data publicly available, allowing for developers to use the data to build applications and tools (e.g., for your phone, computer, or refrigerator) that can be used by the general populous. For example, the reason you can access Google Maps on various websites is because developers used the Google Maps API to build that functionality.

Of course, there are plenty of companies or foundations which will likely never share their data. In these cases though, you can usually find an API that is *related* to that website, or someone may have built (or, they are building) a third-party API for that purpose. 

<font color='darkorange'>**Caveat:** People typically design their APIs such that they decide exactly which kinds of data they want to share. So, they often choose not to share their most lucrative and desirable data. In those cases, you are usually asked to pay some fee.</font>

## APIs vs. Web Scraping

It seems only natural that we should be able to extract any data from the internet by programmatically saving information from web pages we visit. (In fact, it is [perfectly legal](https://melaniewalsh.github.io/Intro-Cultural-Analytics/04-Data-Collection/01-User-Ethics-Legal-Concerns.html).) One way to do this is using [web scraping](https://melaniewalsh.github.io/Intro-Cultural-Analytics/04-Data-Collection/02-Web-Scraping-Part1.html), where you write an algorithm which parses website content, saves data, and loops through several HTML web pages. This method is becoming less effective over the years, as websites are becoming more complex and harder to scrape. Most companies are transitioning to a platform where their data is more easily accessible (and controlled) in an API. Further, writing an API solution is typically going to be more stable than an ad hoc web scraping solution when it comes to data analytics.

Web scraping is typically a last resort for data analysis, but it might be useful for ad-hoc text extraction, or if you find that you have no other options. In these cases, here are some recommended resources:

- First, make sure the data you need can't be extracted using [`pandas.read_html`](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html).
- Melanie Walsh's Tutorial on Web Scraping and BeautifulSoup ([Part 1](https://melaniewalsh.github.io/Intro-Cultural-Analytics/04-Data-Collection/02-Web-Scraping-Part1.html#) and [Part 2](https://melaniewalsh.github.io/Intro-Cultural-Analytics/04-Data-Collection/03-Web-Scraping-Part2.html)) is a great place to start.
- [Scrapy](https://www.scrapy.org/) is an extensive library for more advanced scraping jobs, and their [tutorial](https://docs.scrapy.org/en/latest/intro/tutorial.html) is not bad.
- [Selenium](https://www.selenium.dev/documentation/webdriver/getting_started/) is a web driver, used when websites require interaction (e.g., clicking, typing, etc.) to access data.

## Using Environment Variables

We will discuss environment variables more in a future lesson, but to use APIs properly, we need to have at least a basic understanding of what environment variables are.

When working on any data science project (e.g., like the web app you'll build later in this course), you will likely track your progress using Git/GitHub. But, <font color='darkorange'>keys and secret strings (like the ones we will use to access an API) should never be pushed to GitHub</font>. Instead, it's a best practice to use environment variables when dealing with this kind of data. In short, environment variables are values stored in a special file on your local computer, or on the cloud where your project may be hosted. In this way, those variables are only accessible to agents with access to that file (e.g., your Python interpreter, or the one on the cloud).

In this class, we will use [dotenv](https://github.com/theskumar/python-dotenv#getting-started) to manage environment variables for API. You'll need to create a file with the name *.env* (notice the period) in your project directory to hold any keys or secrets. Since we're going to use this package in this notebook, we'll import the library here. *Note: If you're using Git/GitHub, make sure ".env" is added to your [.gitignore file](https://www.atlassian.com/git/tutorials/saving-changes/gitignore)*.

In [1]:
from dotenv import load_dotenv

## Accessing an API

The steps to access *any* API are about the same, no matter the API. So, in this lesson, we're going to use the [Genius](https://genius.com/) API to access data about songs.

*Note: The Genius API is just one of [many public APIs](https://github.com/public-apis/public-apis) that are equally easy to access.*

### Step 1: Client Access Token

Typically, to use an API, you need a special API key usually called a "Client Access Token", which is kind of like a password. Many APIs require authentication keys to gain access to them. To get your necessary Genius API keys, follow these steps:

1. Navigate to the [api-clients page](https://genius.com/api-clients) (which will prompt you to [sign up for an account](https://genius.com/signup_or_login) if you haven't already). Then, click the button that says **"New API Client"**.
2. Remember, APIs are expecting *developers* to use their APIs to build applications (e.g., iPhone/Android apps, IoT tech, etc.). But, since we're only doing data analysis for a college course in informatics, we only need to fill in the fields for "App Name" (e.g., *"Song Lyrics Project"*), and "App Website URL" (e.g., *"https://github.com/leontoddjohnson/myrepo"*). Then, click **Save**.
3. When you click "Save," you'll be given a series of API Keys: a "Client ID" and a "Client Secret." **Copy/Paste these values into your *.env* file** without quotations, as instructed in the dotenv documentation. For example, my *.env* file looks something like this:
   
```
CLIENT_ID=asdfghjkl;123456789
CLIENT_SECRET=qwertyuiop098765432   
```
    
4. To generate your "Client Access Token," which is the API key that we'll be using in this notebook, you need to click "Generate Access Token". Place that in your *.env* file as you did the other variables, maybe under the variable name ACCESS_TOKEN.

We can access our `ACCESS_TOKEN` by using *dotenv* to load our environment variables into the current environment, then with the built-in Python *os* library to access it.

In [2]:
load_dotenv()

True

In [3]:
import os

In [4]:
# do not print this variable anywhere if the notebook is going on GitHub
ACCESS_TOKEN = os.environ['ACCESS_TOKEN']

### Step 2: Making an API Request

Making an API request is very similar to accessing a URL in your browser. But, instead of getting a rendered HTML web page in return, you get some data in return.

There are a few different ways that we can [query the Genius API](https://docs.genius.com/#songs-h2), but here we'll use [the basic search](https://docs.genius.com/#search-h2), which allows you to get a bunch of Genius data about any artist or songs that you search for:

```http://api.genius.com/search?q={search_term}&access_token={client_access_token}```

*Note: This is a typical API call. In the URL, `q=` usually indicates a **query**, and the `&` separates different arguments or headings to pass on to the API server.*

First we're going to assign the string "Missy Elliott" to the variable `search_term`. Then we're going to make an f-string ("formatted" string) URL that contains the variables we'd like to include in our query.

In [5]:
search_term = "Missy Elliott"

In [6]:
genius_search_url = f"http://api.genius.com/search?q={search_term}&access_token={ACCESS_TOKEN}"

You can see the data we'll be requesting from this API by printing the `genius_search_url`, pasting it into your browser.

In [8]:
print(genius_search_url)

http://api.genius.com/search?q=Missy Elliott&access_token=os.environ['ACCESS_TOKEN']


The data you might see when you navigate to your URL is in [JSON](https://www.w3schools.com/whatis/whatis_json.asp) format. JSON is an acronym for JavaScript Object Notation, and it is a data format commonly used by APIs. JSON data can be nested, and contains key-value pairs, much like a Python dictionary.

We can access this JSON directly in Python using the [`requests` library](https://requests.readthedocs.io/en/latest/) to send HTTP requests to a remote server. If you like, you can **read more about [HTTP requests](https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview) and [HTTP messages](https://developer.mozilla.org/en-US/docs/Web/HTTP/Messages)**, but it suffices to say that a request represents an online communication between your computer and the server storing the data you want.

In particular, we will be making a GET request to *"get"* data from the Genius server. This is one of the [common HTTP request types](https://blog.postman.com/what-are-http-methods/), where the other types are *rarely* used in data science work.

In [8]:
import requests

In [9]:
# here, we make a "GET" request to the Genius server
response = requests.get(genius_search_url)
json_data = response.json()

In [10]:
json_data

{'meta': {'status': 401,
  'message': 'This call requires an access_token. Please see: https://genius.com/developers'}}

More formally, we can (and should) place our authorization credentials in an [Authorization header](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Authorization). The Genius API documentation indicates `"Bearer"` as the authorization scheme. 

In [11]:
genius_search_url = f"http://api.genius.com/search?q={search_term}"

response = requests.get(genius_search_url, 
                        headers={"Authorization": "Bearer " + ACCESS_TOKEN})
json_data = response.json()

In [12]:
json_data

{'meta': {'status': 401,
  'message': 'This call requires an access_token. Please see: https://genius.com/developers'}}

One way to navigate this JSON object is to use the `.keys()` method for dictionaries, iteratively.

In [13]:
json_data.keys()

dict_keys(['meta'])

In [14]:
json_data['response'].keys()

KeyError: 'response'

In [15]:
len(json_data['response']['hits'])

KeyError: 'response'

We can also use the `json` package to save a temporary *.json* file, open the file and [format it](https://code.visualstudio.com/docs/languages/json#_formatting) in VS Code, then navigate it using the breadcrumbs at the top of the page.

In [16]:
import json

# note: this file was added to the gitignore
with open('temp.json', 'w') as f:
    json.dump(json_data, f)

Genius places all of its search results into the "hits" element. By default, it looks like it returns at most 10 search results for any request. Also, according to the documentation, we can use [referents](https://docs.genius.com/#referents-h2) to increase that number to a maximum of 20 results per request using `per_page`. With this slight adjustment added, let's consolidate our request into a single function to use again later.

In [1]:
def genius(search_term, per_page=15):
    '''
    Collect data from the Genius API by searching for `search_term`.
    
    **Assumes ACCESS_TOKEN is loaded in environment.**
    '''
    genius_search_url = f"http://api.genius.com/search?q={search_term}&" + \
                        f"access_token={ACCESS_TOKEN}&per_page={per_page}"
    
    response = requests.get(genius_search_url)
    json_data = response.json()
    
    return json_data['response']['hits']

In [18]:
json_data = genius("The Beatles")
len(json_data)

KeyError: 'response'

## Loading JSON Data Into a DataFrame

For us to efficiently work with the JSON data, we need to load them into a DataFrame. Conveniently, our "JSON" data is actually now stored in a Python list (of dictionaries), and this is exactly one of the main formats that `pd.DataFrame` [expects](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html#pandas-dataframe).

*Note: JSON data is often saved in files, much like CSVs. In these cases, you can use `pd.read_json` to read in those files.*

In [19]:
import pandas as pd

In [20]:
l = [
    {'a': 1, 'b': 2},
    {'a': 3, 'b': 4}
]

In [21]:
pd.DataFrame(l)

,a,b
0,1,2
1,3,4


In [22]:
json_data[0]

KeyError: 0

When we look at any of the hits, we see the data we're interested in is contained in the `"result"` element. We can consolidate all of the "result" elements for each "hit" using a list comprehension.

**Looking ahead:** Notice that the `"stats"` and the `"primary_artist"` elements contain *dictionaries* of interesting data that we'll need to unpack once we have our data into a DataFrame.

In [23]:
hits = [hit['result'] for hit in json_data]
df = pd.DataFrame(hits)

TypeError: string indices must be integers, not 'str'

In [24]:
df.head()

NameError: name 'df' is not defined

Recall that `"stats"` and `"primary_artist"` contain dictionaries which we want to unpack. We can [use](https://stackoverflow.com/a/38231651) `pd.apply(pd.Series)` and `pd.concat` to explode these into columns. We'll need to make a slight adjustment to the column names to avoid repeats.

In [25]:
# df['stats']
df['stats'].iloc[0]

NameError: name 'df' is not defined

In [26]:
df_stats = df['stats'].apply(pd.Series)
df_stats.rename(columns={c:'stat_' + c for c in df_stats.columns},
                inplace=True)

df_stats.head()

NameError: name 'df' is not defined

In [27]:
df_primary = df['primary_artist'].apply(pd.Series)
df_primary.rename(columns={c:'primary_artist_' + c for c in df_primary.columns},
                  inplace=True)
df_primary.head()

NameError: name 'df' is not defined

In [28]:
# we maintain the original column, and "concat/merge" on the **shared index**
df = pd.concat((df, df_stats, df_primary), axis=1)

NameError: name 'df' is not defined

Let's consolidate all of this into one function which creates a dataframe of genius data given a search term.

In [29]:
def genius_to_df(search_term, n_results_per_term=10):
    json_data = genius(search_term, per_page=n_results_per_term)
    hits = [hit['result'] for hit in json_data]
    df = pd.DataFrame(hits)

    # expand dictionary elements
    df_stats = df['stats'].apply(pd.Series)
    df_stats.rename(columns={c:'stat_' + c for c in df_stats.columns},
                    inplace=True)
    
    df_primary = df['primary_artist'].apply(pd.Series)
    df_primary.rename(columns={c:'primary_artist_' + c for c in 
                               df_primary.columns},
                      inplace=True)
    
    df = pd.concat((df, df_stats, df_primary), axis=1)
    
    return df

### Collecting Multiple API Calls

We are going to want to perform analysis on more than one artist, so let's use what we've written above to collect data from multiple API calls by looping through multiple search terms. When we loop through each search term, we use the [tqdm package](https://pypi.org/project/tqdm/) to help us visualize our progress (you may need to use `pip` to install it). This kind of thing is helpful when we're running multiple API calls, and we don't know how long it will take.

In [30]:
from tqdm import tqdm

In [31]:
search_terms = ['The Beatles', 'Missy Elliot', 'Andy Shauf', 'Slowdive', 'Men I Trust']
n = 10

dfs = []

# loop through search_terms in question
for search_term in tqdm(search_terms):
    df = genius_to_df(search_term, n_results_per_term=n)
    
    # add to list of DataFrames
    dfs.append(df)

  0%|          | 0/5 [00:00<?, ?it/s]


KeyError: 'response'

In [32]:
df_genius = pd.concat(dfs)

ValueError: No objects to concatenate

In [33]:
df_genius.shape

NameError: name 'df_genius' is not defined

In [34]:
df_genius.columns

NameError: name 'df_genius' is not defined

In [35]:
df_genius.sample(3)

NameError: name 'df_genius' is not defined

## Using an API Wrapper

More often than not, someone has built an "API Wrapper" for the API you are working with. An API wrapper makes an API easier to use, and it often extends the API itself. It will typically consist of classes and functions similar to the ones we've built above, but spanning a wide range of functionality and access to the API. For example, John Miller's [LyricsGenius](https://github.com/johnwmillr/LyricsGenius) gives us an almost universal access to the Genius website, and it even uses web scraping to collect song lyrics themselves.

<span style="color: lightblue">**If ever you're working with an API, do some Googling to make sure there isn't a wrapper you can use to make things easier on you!**</span>

# Python Scripts

Suppose we want to extract a good bit of data from the Genius API (or any other API for that matter), but we want to run the script on our computer while we do other processes in Jupyter (or sleep). Or, consider having several Python-based "jobs" we want to run on a scheduled basis, where we don't want the responsibility of opening Jupyter and typing "Shift-Tab" every time. In these cases, we'll want to run executable code in the form of a [Python Script](https://realpython.com/run-python-scripts/).

In short, a Python script is a Python file that runs a particular routine on a [thread](https://www.liquidweb.com/blog/difference-cpu-cores-thread/) of your CPU, line by line until it's over. You can run these Python files in several different ways, including:

- **from the terminal**
- from an interactive Python shell (e.g., a notebook)
- as a periodically *scheduled* "job"

In this class, we'll talk about running Python files as scripts from the terminal. Running scripts from IPython can cause issues, not the least of which is the script output is relegated to a notebook cell which can take up unnecessary space in the notebook. Whereas, periodically scheduled jobs are best run on virtual instances hosted in a cloud computing environment, which is outside the scope of this class.

## Creating Python Scripts

All we need to do to create a script is to make a .py file, and place our executable code within the file. It's best to keep these files in the "root" directory of your project (e.g., the same directory-level as this notebook), and **use short names with the `snake_case` naming convention.**

For example, notice that the functions we've defined above are all also defined within the *genius_api.py* file in this directory. In that file, we do our best to follow the [PEP-8](https://pep8.org/) guidelines.

### **kwargs

You'll notice the use of `**kwargs` in the `genius_to_dfs` function in the Python file. This is in reference to "keyword arguments" which can be passed from a function to "sub"-function. Technically, the `**` at the beginning can be used to unpack sets or dictionaries:

In [36]:
my_dict_1 = {'a': 1, 'b': 2}
my_dict_2 = {'c': 3, 'd': 4}

In [37]:
{**my_dict_1, **my_dict_2}

{'a': 1, 'b': 2, 'c': 3, 'd': 4}

But, it can also be used to pass arguments:

In [38]:
def called(y=3, z=5):
    print(y + z)

In [39]:
def calling(x, **kwargs):
    print("the 'x' value is ", x)
    print("\nand the sum of interest is:")
    called(**kwargs)

In [40]:
calling(4, y=5)

the 'x' value is  4

and the sum of interest is:
10


### Modules

Functions exist as a sort of short-hand: reusable Python structures which help us to avoid writing repetitive code blocks. I.e., instead of writing the same block of code multiple times, we simply call a single function — it saves room and keeps things clean. In the same way that functions (and classes) can be imported from third-party (or built-in) packages, they can also be imported from modules we build ourselves!

In its most basic form, a [Python module](https://realpython.com/python-modules-packages/#python-modules-overview) is a Python file used to maintain Python objects (e.g., functions, variables, etc.). You can "import" your reusable code using the `import` statement as you would any other Python module.

**It is a good practice to build and test out your functions in Jupyter, and when you feel comfortable with them, *move* them into your Python module.** Use Jupyter notebook *as a notebook*, and keep any reusable code (e.g., functions) in the Python file. Whenever you're working on a project, keep open your Jupyter notebook for building and testing code, and keep your IDE (e.g., VS Code) open for updating your functions.

In [41]:
from genius_api import testing

Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


In [42]:
testing()

Testing 1, 2, 3 ...


Notice that if we make changes to the file, this function will not update the output. This is because the code that is "imported" into the current Python interpreter is static, from the most recent import call. We can either "re-import" the module, or we can use the `reload` function from `importlib`. You can also use 

In [43]:
from importlib import reload

In [44]:
# we need to import the module itself first
import genius_api

In [45]:
# run this cell after each change
reload(genius_api)
from genius_api import testing

Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


In [46]:
testing()

Testing 1, 2, 3 ...


Alternatively, you can use the [autoreload](https://ipython.org/ipython-doc/3/config/extensions/autoreload.html) functionality in Jupyter to make sure updates to our code reflect in the notebook.

In [47]:
# this only needs to be run once, maybe at the top of a notebook
%load_ext autoreload

In [48]:
# notice these share the same cell; create one cell per module reference
%autoreload 2
from genius_api import testing

In [49]:
testing()

Testing 1, 2, 3 ...


### \_\_name__

You'll notice at the bottom of the Python file, there is a reference to `__name__` and the string `"__main__"`. [In short](https://realpython.com/if-name-main-python/), every time a Python module is accessed, it is given a name. And, according to the [Python documentation](https://docs.python.org/3/library/__main__.html):

*`__main__` is the name of the environment where top-level code is run. "Top-level code" is the first user-specified Python module that starts running. It’s "top-level" because it imports all other modules that the program needs. Sometimes “top-level code” is called an "entry point" to an application.*

In fact, the `__name__` is stored as a global variable for the environment. For example:

In [50]:
__name__

'__main__'

This can happen when Python is called from *"the scope of an interactive prompt"* (e.g., in Jupyter notebook), and it can also happen when a Python file is called from your computer's terminal using `python filename.py`.

*Note: this is **not** the same as the `!`-defined terminal in a Jupyter notebook*.

In [51]:
# uncomment the ... print("__name__ is" ... line in the module
# !python genius_api.py

*Note: you may find that there are package install errors, since the `!` is commanding from the environment that Jupyter was called from (i.e., where you ran `jupyter lab`, which might not be the same as the notebook kernel.*

Otherwise, the `__name__` is the name of the module itself.

In [52]:
# the imported module
genius_api.__name__

'genius_api'

In [53]:
from genius_api import NAME_DEMO

In [54]:
NAME_DEMO

'genius_api'

In summary:

- When you **call a module** in an `import` statement, `__name__ == "__main__"` is `False`.
- When you **call a script** from a "top-level environment", `__name__ == "__main__"` is `True`.

## Running Python Scripts

### Defining the environment

Before you run a Python script in your terminal, you need to *source* an environment so that the following `python` commands instantiate a Python instance with those packages available. Recall that with uv (and venv), your environment is sourced from the local *.venv* directory. To source this environment, **first make sure you are in the project directory.** Then, run the following command (as described [here](https://docs.astral.sh/uv/pip/environments/#using-a-virtual-environment)):

```bash
source .venv/bin/activate
```

*Note: If you want to get fancy, there are ways to [alias this command](https://linuxize.com/post/how-to-create-bash-aliases) making it easier for you in the future.*

After running any script(s), you can deactivate your environment with the `deactivate` command.

### Run from the terminal

We run a script as a "command" from the command line:

```bash
python <myfilename>.py
```

Python will spin up a process, run the lines of code, and kill the process.

<font color='darkorange'>**Recommendation**: If you are extracting data with a function like `genius_to_df` multiple times (e.g., like the loop used in `genius_to_dfs`), it is *highly* recommended that you save data iteratively, in case you run into errors. See the function in the *genius_api.py* file for an example of this.</font>


### Run a background process

In many cases, you'll want a script to run on your local computer "in the background" so that when your computer goes to sleep, or if you close a terminal, the script is still running.

```bash
nohup python <command> &
```

Don't forget the `&` at the end. This directs the process to be run in the background. The [`nohup` command](https://www.ibm.com/docs/en/aix/7.3.0?topic=n-nohup-command#nohup__title__4) directs output to a *nohup.out* file in the same (working) directory.

### Check Running Processes

```bash
ps ax
ps ax | grep <script_name> 
```

The former [`ps` command](https://www.ibm.com/docs/en/aix/7.3.0?topic=p-ps-command#ps__title__4) (with [options](https://www.ibm.com/docs/en/aix/7.3.0?topic=p-ps-command#ps__title__7) `a` and `x`) will list the running processes, and the latter will [pipe](https://www.geeksforgeeks.org/linux-unix/piping-in-unix-or-linux/) the results to the [`grep` command](https://www.ibm.com/docs/en/aix/7.3.0?topic=g-grep-command#grep__title__4) to match on running processes with `<script_name>` in its name.

### Kill a Process

If you're done with the script, run

```bash
kill <process_id>
```

To kill the process with `<process_id>` in the first column from the `ps` command, above. **Be careful with this!**

## Parallel Computing (optional)

Consider the fact that your computer likely has multiple "cores". I.e., when you buy a "quad-core" computer, this means that your CPU contains 4 *separate* processing units. **Each processing unit (core) can run Python routines independently.** This means that you can parallelize big operations across all your cores if you can harness each one individually. Python does not automatically dole out sub-processes based on our code, so we need to direct the computer to do so explicitly. We do this with [parallel computing](https://pythonnumericalmethods.studentorg.berkeley.edu/notebooks/chapter13.01-Parallel-Computing-Basics.html).

The [multiprocessing Pool class](https://docs.python.org/3/library/multiprocessing.html) is a straight-forward way to employ computing parallelization. The simplest example (illustrated in that first code block in the link) instantiates a **pool of workers**, where each worker completes an assigned task, and returns the results. You'll notice that only top-level code can create the pool of workers, so we need the `__name__` to be `__main__`.

For example, the `SIMPLE MULTIPROCESSING EXAMPLE` and the `API MULTIPROCESSING EXAMPLE` in the *genius_api.py* file show how you can use multiprocessing to speed up processes. Uncomment one of them, and run the script from your terminal (within the proper environment):

```bash
python genius_api.py
```

You'll notice that there are several different processes defined (by their Process ID, `pid`), and each one runs the `job_test` function separately. Each process is assigned a **thread** which employs your CPUs (in a way, the CPUs are "multi-tasked" across threads ...). If you want to know how many threads are available on your machine, you can use `psutil` or `os`.

In [55]:
import psutil
import os

In [56]:
# physical cores
print("core count:\t", psutil.cpu_count(False))

# threads available
print("thread count:\t", psutil.cpu_count(True))

# # threads available (same thing)
print("*thread count:\t", os.cpu_count())

core count:	 12
thread count:	 12
*thread count:	 12


### joblib

Another useful tool for multiprocessing is [joblib](https://pythonnumericalmethods.studentorg.berkeley.edu/notebooks/chapter13.03-Use-joblib.html). This tool is nice because it provides a relatively simple interface for multiprocessing/multithreading operations, *and* it integrates well with other data science tools you'll see later on.

**A few caveats:**

- The [joblib syntax and documentation](https://joblib.readthedocs.io/en/stable/) can tend to be a bit obscure; read it carefully.
- Typicaly, any joblib integration with other data science tools is usually handled by the tool in question. So, in most cases, you won't need to deal with it directly.
- The [default `loky` backend](https://joblib.readthedocs.io/en/latest/parallel.html#thread-based-parallelism-vs-process-based-parallelism) can be finicky ...
  - If your function can work on a *single* thread (usually the case for basic Python functions), use the `backend="threading"`.
  - Otherwise, use the `backend="multiprocessing"` which references the *multiprocessing* package described above.


In [57]:
from joblib import Parallel, delayed
from genius_api import job_test

The `job_test` function does the following:

1. Take in a numerical value `x` and an *optional* `mult` (default 2)
2. Print `x` and the process ID associated with the current process thread.
3. Wait a random amount of time between $1\pm 0.5$ seconds.
4. Return `x * mult`.

In [58]:
job_test(3)

PID: 8720 ... num: 3


6

We can parallelize this function and save the results using joblib.

In [59]:
# this example uses the multiprocessing backend
results = Parallel(n_jobs=8, backend='multiprocessing', verbose=1)(
    delayed(job_test)(i) for i in range(10)
)

[Parallel(n_jobs=8)]: Using backend MultiprocessingBackend with 8 concurrent workers.
Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


PID: 8780 ... num: 0


Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


PID: 8775 ... num: 1


Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


...

Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


PID: 8778 ... num: 6


Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


PID: 8776 ... num: 7
PID: 8780 ... num: 8
PID: 8774 ... num: 9


[Parallel(n_jobs=8)]: Done   6 out of  10 | elapsed:    3.4s remaining:    2.2s
[Parallel(n_jobs=8)]: Done  10 out of  10 | elapsed:    4.1s finished


In [60]:
results

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

In [61]:
# this example uses threading, and adjusts the `mult` parameter
results = Parallel(n_jobs=8, backend='threading', verbose=1)(
    delayed(job_test)(i, 3) for i in range(10)
)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


PID: 8720 ... num:PID: 8720 ... num: 1
 0
PID: 8720 ... num: 2
PID: 8720 ... num: 3
PID: 8720 ... num: 4
PID: 8720 ... num: 5
PID: 8720 ... num: 6
PID: 8720 ... num: 7
PID: 8720 ... num: 8
PID: 8720 ... num: 9


[Parallel(n_jobs=8)]: Done   6 out of  10 | elapsed:    1.5s remaining:    1.0s
[Parallel(n_jobs=8)]: Done  10 out of  10 | elapsed:    1.8s finished


In [62]:
results

[0, 3, 6, 9, 12, 15, 18, 21, 24, 27]

*In general, **parallel processing is only beneficial when you are running a routine over a large amount of data** (this is clear in the `backend="multiprocessing"` example, above). If there is not enough data, then the time it takes to "spin up" each process may end up [overshadowing](https://pythonnumericalmethods.studentorg.berkeley.edu/notebooks/chapter13.02-Multiprocessing.html#visualize-the-execution-time) any sort of gains in efficiency you might get from the processing itself.*

# Explore

Test your understanding of this week's content with the following exploration(s).

*Note: unless otherwise noted, **explorations are completely optional and will not be reviewed.***

## Exploration 1

Write a Python script (i.e., an Python .py file with executable lines of code at the bottom) to retrieve data for a *list* of search terms from **some other** API (i.e., NOT the Genius API).

- Implement error handling using `try`-`except` blocks to handle potential issues such as API rate limits, response errors, or network errors.
- Build your script to save the results to a CSV.

*Note: you're welcome to use the genius_api.py file as a reference.*

## Exploration 2

Try to incorporate multiprocessing into Exploration 1, and save your data to a CSV file (or separate CSV files).